In [1]:
import os
import sys
import time
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt

# ray 관련 라이브러리 임포트
import ray
from ray import tune
from ray.tune.registry import register_env

# 프로젝트 루트 디렉토리 설정
os.chdir("/home/youngjins/project/belief_trading")

# lib 하위 경로 추가
sys.path.append("/home/youngjins/project/belief_trading/lib/")
sys.path.append("/home/youngjins/project/belief_trading/lib/abides_jpmc_public")

# 환경 임포트
import gym
import abides_gym
from abides_gym.envs.markets_execution_environment_v0 import (
    SubGymMarketsExecutionEnv_v0,
)

# lib 모듈 임포트
from abides_core import abides
from abides_core.utils import parse_logs_df, ns_date, str_to_ns, fmt_ts
from abides_markets.configs import rmsc04
from lib.utils import gym_types, flatten_dict

In [2]:
# 매개 변수 선언
seed = 0
gym_type_str = "markets-execution-v0"

# 함수 init
ray.shutdown()
ray.init()

np.random.seed(seed)

gym_type_abb = gym_types[gym_type_str]

register_env(
    gym_type_str,
    lambda config: SubGymMarketsExecutionEnv_v0(**config),
)

config = rmsc04.build_config()

2025-03-17 17:22:11,689	INFO worker.py:1752 -- Started a local Ray instance.


In [3]:
# policy 정의
class policyPassive:
    """
    패시브 정책은 항상 1을 반환 (시장 가격에 영향을 최소화, 소극적으로 주문을 실행하는 전략)
    """

    def __init__(self):
        self.name = "passive"

    def get_action(self, state):
        return 1


class policyAggressive:
    """
    공격적 정책은 항상 0을 반환 (적극적으로 주문을 처리하는 전략, 시장 충격이 클 수 있음)
    """

    def __init__(self):
        self.name = "aggressive"

    def get_action(self, state):
        return 0


class policyRandom:
    """
    무작위 정책은 0과 1 중에서 무작위로 선택 (패시브와 공격적 전략을 무작위로 혼합하는 방식)
    """

    def __init__(self):
        self.name = "random"

    def get_action(self, state):
        return np.random.choice([0, 1])


class policyRandomWithNoAction:
    """
    완전 무작위 정책은 0, 1, 2 중에서 무작위로 선택 (관망하며 시장 상황을 지켜보는 옵션을 포함)
    """

    def __init__(self):
        self.name = "random_no_action"

    def get_action(self, state):
        return np.random.choice([0, 1, 2])

In [4]:
# 강화학습 관련 변수 선언
env = gym.make(
    gym_type_str,
    background_config="rmsc04",
    timestep_duration="10S",
    execution_window="04:00:00",
    parent_order_size=20000,
    order_fixed_size=50,
    not_enough_reward_update=-100,  # penalty
)

env.seed(seed)

target_agent = policyRandomWithNoAction()

In [ ]:
# 시뮬레이션 시작
state = env.reset()
done = False
episode_reward = 0

# 1개 에피소드 실행 (소요 시간 측정)
start_time = time.time()    
while not done:
    action = target_agent.get_action(state)
    state, reward, done, info = env.step(action)
    episode_reward += reward

end_time = time.time()
print(f"Execution time: {end_time - start_time} seconds")

# could add a few more...
output = flatten_dict(info)
output["episode_reward"] = episode_reward
output["name"] = target_agent.name

KeyboardInterrupt: 